# DL ·Lecture 12 — Encoder-decoder & Representation Learning

You've done the by-hand pieces; now **train the real thing on real data**. We build an **autoencoder** on MNIST, watch a **bottleneck** force a compressed representation, read the latent as an **embedding** (cosine similarity, nearest neighbours), and **probe + reuse** the encoder for classification — the seed of representation learning that underlies today's foundation models.

Follows the course rhythm: **setup/GPU-check → get the data → look at the data → build (🔧) → train (🔧) → evaluate → experiment → reflect**. Real `torch`/`torchvision` — datasets download at runtime, no `pip install`.

> **Run on a GPU:** Colab → **Runtime → Change runtime type → T4 GPU** (it works on CPU too, just slower).

## 0. Setup & GPU check

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from sklearn.decomposition import PCA

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU for the intended "
          "~2-3 min run (it still works on CPU, just slower).")

# Colab-friendly sizes; raise N_TRAIN to 60000 and EPOCHS for sharper results on a GPU.
N_TRAIN, EPOCHS, BATCH = 12000, 6, 256

## 1. The encoder-decoder pattern

Many deep models share one shape: two networks joined at a narrow waist.

```
x  ──encoder──▶  z  ──decoder──▶  x̂
              (bottleneck)
```

- The **encoder** maps input `x` to a small vector `z` (the **latent** / **code** / **representation**) — it discards detail, keeps structure.
- The **decoder** maps `z` back out to a target: a reconstruction, a mask, a translation.

Autoencoders (this lecture), U-Nets, and seq-to-seq translation are all this skeleton — *squeeze to a latent, expand from it*. What changes is only what the decoder produces.

## 2. Get the data — and look at it

An **autoencoder**'s target *is its own input*: reconstruct `x` from a latent `z` computed from `x`, minimising `distance(x, x̂)`. **No labels** — the signal is the input itself (**self-supervised**). We use MNIST (28×28 = 784 pixels, values in [0,1]).

In [ ]:
tfm = transforms.ToTensor()                                    # -> float tensor in [0, 1]
train_full = datasets.MNIST(root="./data", train=True,  download=True, transform=tfm)
test_full  = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_ds = torch.utils.data.Subset(train_full, range(N_TRAIN))  # subset for speed
loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True)

# ALWAYS look at the data first
imgs, labels = next(iter(torch.utils.data.DataLoader(train_full, batch_size=10)))
fig, ax = plt.subplots(1, 10, figsize=(12, 1.6))
for a, im, y in zip(ax, imgs, labels):
    a.imshow(im.squeeze(), cmap="gray"); a.set_title(int(y)); a.axis("off")
plt.suptitle("MNIST — the network must reconstruct these from a tiny latent"); plt.show()

## 3. Build the autoencoder (🔧)

Two `nn.Sequential` blocks joined at a **32-dim bottleneck**. One important choice from the reading: MNIST pixels are in [0,1], so the right loss is **binary cross-entropy**, not MSE. The clean, numerically stable way is to have the decoder output **bare logits** (no final `Sigmoid`) and use **`nn.BCEWithLogitsLoss`**, which fuses the sigmoid into the loss. (Sigmoid + `MSELoss` — the naive pairing — trains *weakly*: the sigmoid's gradient is tiny when saturated.)

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, in_dim=784, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(),
            nn.Linear(128, latent_dim),           # <- the bottleneck
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, in_dim),               # bare LOGITS (no Sigmoid; BCEWithLogitsLoss adds it)
        )

    def forward(self, x):
        z = self.encoder(x)                       # (batch, 784) -> (batch, latent)
        logits = self.decoder(z)                  # (batch, latent) -> (batch, 784)
        return logits, z

def train_ae(model, loader, epochs, lr=1e-3, quiet=False):
    model.to(device).train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()              # [0,1] targets -> BCE, not MSE
    hist = []
    for ep in range(epochs):
        total, nb = 0.0, 0
        for x, _ in loader:                       # labels ignored — self-supervised
            x = x.view(x.size(0), -1).to(device)  # flatten to (batch, 784)
            logits, _ = model(x)
            loss = loss_fn(logits, x)             # reconstruct the INPUT
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); nb += 1
        hist.append(total / nb)
        if not quiet:
            print(f"epoch {ep+1}/{epochs}  BCE {hist[-1]:.4f}")
    return hist

model = AutoEncoder(latent_dim=32)
print(model)

## 4. Train it (🔧) and look at the reconstructions

In [ ]:
hist = train_ae(model, loader, epochs=EPOCHS)
plt.plot(range(1, len(hist) + 1), hist, "o-")
plt.xlabel("epoch"); plt.ylabel("BCE reconstruction loss"); plt.title("autoencoder training"); plt.show()

In [ ]:
# top row: original test digits; bottom row: the model's reconstruction (sigmoid of the logits)
model.eval()
test_imgs = torch.stack([test_full[i][0] for i in range(10)]).view(10, -1).to(device)
with torch.no_grad():
    logits, _ = model(test_imgs)
    recon = torch.sigmoid(logits)                 # logits -> pixels in [0,1] for viewing

fig, ax = plt.subplots(2, 10, figsize=(12, 2.6))
for i in range(10):
    ax[0, i].imshow(test_imgs[i].cpu().view(28, 28), cmap="gray"); ax[0, i].axis("off")
    ax[1, i].imshow(recon[i].cpu().view(28, 28),     cmap="gray"); ax[1, i].axis("off")
ax[0, 0].set_ylabel("input", rotation=0, ha="right"); ax[1, 0].set_ylabel("x̂", rotation=0, ha="right")
plt.suptitle("top: input   bottom: reconstruction from a 32-dim latent"); plt.show()

## 5. The bottleneck is the whole point

If dim(z) ≥ dim(x) the network can just **copy** the input (≈0 error, learns nothing — an expensive copy machine). Force dim(z) < dim(x) and perfect copying is impossible, so the encoder must find the **regularities** shared across examples. First, the reading's tiny hand example — `x = [2, 4, 6, 10]` at three bottleneck widths:

In [ ]:
x = torch.tensor([2., 4., 6., 10.])
# width 4: identity (copies exactly) ; width 2: each pair -> its average ; width 1: overall mean
recons = {
    "width 4 (copy)": x,
    "width 2 (pair means)": torch.tensor([3., 3., 8., 8.]),
    "width 1 (overall mean)": torch.full((4,), x.mean().item()),
}
for name, xh in recons.items():
    print(f"{name:24} x̂ = {xh.tolist()}   MSE = {((x - xh) ** 2).mean():.2f}")
print("\nNarrower z -> more compression -> more error. Width 4 'wins' (MSE 0) but learned nothing.")

Now the same effect on MNIST: train autoencoders at several bottleneck widths and plot the reconstruction error. It falls as the latent widens — and the knee tells you roughly how many dimensions the data really needs:

In [ ]:
widths = [2, 8, 16, 32, 64]
mse_by_width = []
eval_x = torch.stack([test_full[i][0] for i in range(1000)]).view(1000, -1).to(device)
for w in widths:
    torch.manual_seed(w)                                         # fixed init per width -> stable curve
    m = AutoEncoder(latent_dim=w)
    train_ae(m, loader, epochs=max(3, EPOCHS - 2), quiet=True)   # a bit shorter, per width
    m.eval()
    with torch.no_grad():
        recon = torch.sigmoid(m(eval_x)[0])
    mse_by_width.append(F.mse_loss(recon, eval_x).item())
    print(f"latent dim {w:3d}  ->  test recon MSE {mse_by_width[-1]:.4f}")

plt.plot(widths, mse_by_width, "o-")
plt.xlabel("bottleneck width  dim(z)"); plt.ylabel("test reconstruction MSE")
plt.title("wider bottleneck -> generally lower reconstruction error"); plt.show()

### Linear autoencoder ≈ PCA
A **linear** autoencoder under MSE recovers the same subspace as **PCA**; the ReLU layers above make ours a *nonlinear* generalisation — "learned, nonlinear dimensionality reduction." One fair caveat: PCA(32) is the **optimal linear** reconstruction *for MSE*, while our AE is trained on BCE — so on the MSE yardstick PCA has a built-in edge, and a nonlinear AE only pulls ahead with enough training. Compare the fully-trained latent-32 model against PCA(32) and read the two numbers:

In [ ]:
Xtr = torch.stack([train_full[i][0] for i in range(N_TRAIN)]).view(N_TRAIN, -1).numpy()
Xte = eval_x.cpu().numpy()
pca = PCA(n_components=32).fit(Xtr)
pca_mse = ((Xte - pca.inverse_transform(pca.transform(Xte))) ** 2).mean()

model.eval()                                            # the fully-trained (EPOCHS) latent-32 AE
with torch.no_grad():
    ae_mse = F.mse_loss(torch.sigmoid(model(eval_x)[0]), eval_x).item()

print(f"PCA(32) linear recon MSE  : {pca_mse:.4f}   (optimal *linear* reconstruction for MSE)")
print(f"nonlinear AE(32) recon MSE: {ae_mse:.4f}")
print("PCA is the linear optimum on this metric; the nonlinear AE closes in (and can pass it)")
print("with more epochs — the ReLU depth is what lets it go beyond a purely linear projection.")

## 6. Latents as embeddings — meaning becomes geometry

The latent `z` is an **embedding**: similar inputs land **close together**. We measure closeness with **cosine similarity** — the cosine of the angle between two vectors, (a·b)/(‖a‖‖b‖) — which ignores magnitude and asks only *"do they point the same way?"* (1 = same direction, 0 = unrelated, −1 = opposite). Embed the test digits and do a **nearest-neighbour search** by cosine — the neighbours should be the *same digit*:

In [ ]:
N = 2000
X = torch.stack([test_full[i][0] for i in range(N)]).view(N, -1).to(device)
Y = torch.tensor([test_full[i][1] for i in range(N)])
model.eval()
with torch.no_grad():
    Z = model.encoder(X)                          # (N, 32) latents

def nearest(query_idx, k=5):
    sims = F.cosine_similarity(Z[query_idx:query_idx + 1], Z, dim=1)
    return sims.argsort(descending=True)[1:k + 1]  # skip itself (rank 0)

q = 7
nn_idx = nearest(q)
fig, ax = plt.subplots(1, 6, figsize=(9, 1.8))
ax[0].imshow(X[q].cpu().view(28, 28), cmap="gray"); ax[0].set_title(f"query={int(Y[q])}"); ax[0].axis("off")
for j, idx in enumerate(nn_idx):
    ax[j + 1].imshow(X[idx].cpu().view(28, 28), cmap="gray")
    ax[j + 1].set_title(int(Y[idx])); ax[j + 1].axis("off")
plt.suptitle("cosine nearest neighbours in latent space (should match the query digit)"); plt.show()
match = (Y[nn_idx] == Y[q]).float().mean().item()
print(f"{match*100:.0f}% of the 5 nearest latents share the query's digit label")

### The data decides whether the bottleneck can win
A tiny **2→1→2** autoencoder (no nonlinearity) on **correlated** pairs (a, 2a) drives the MSE to ≈0 — one number genuinely summarises the pair. Make the pairs **unrelated** and it plateaus well above zero: one number simply cannot reconstruct two independent ones. Same model, same loop — the *data* decides.

In [ ]:
def tiny_ae_final_mse(pairs, steps=3000, lr=0.05, seed=0):
    torch.manual_seed(seed)
    Xp = torch.tensor(pairs, dtype=torch.float32)
    net = nn.Sequential(nn.Linear(2, 1), nn.Linear(1, 2))     # 2 -> 1 -> 2, linear
    opt = torch.optim.Adam(net.parameters(), lr=lr); lf = nn.MSELoss()
    for _ in range(steps):
        loss = lf(net(Xp), Xp); opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

print(f"correlated (a, 2a): final MSE = {tiny_ae_final_mse([[1,2],[2,4],[3,6],[4,8]]):.4f}   -> ~0 (1-D structure)")
print(f"unrelated pairs   : final MSE = {tiny_ae_final_mse([[1,4],[2,1],[3,5],[4,2]]):.4f}   -> stuck above 0")

## 7. Probe and reuse the representation

A representation is only useful if it captures structure. Two probes: (1) **visualise** the latents in 2-D and see whether digits cluster; (2) a **linear probe** — freeze the encoder, train a *single* linear layer on top for classification, and read its accuracy. If a linear classifier on `z` does well, the latent has organised the data usably. (We visualise with PCA — a *linear* 2-D projection — for a dependency-free view; t-SNE or UMAP, named in the lecture, would separate the clusters even more sharply.)

In [ ]:
# (1) visualise: reduce the 32-d latents to 2-D with PCA, colour by the true digit
from sklearn.decomposition import PCA
Z2 = PCA(n_components=2).fit_transform(Z.cpu().numpy())
plt.figure(figsize=(6.4, 5.2))
sc = plt.scatter(Z2[:, 0], Z2[:, 1], c=Y.numpy(), cmap="tab10", s=8, alpha=0.7)
plt.colorbar(sc, label="digit"); plt.title("2-D view of the learned latent (colour = digit)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

In [ ]:
# (2) linear probe: FREEZE the encoder, train only a linear head on labels, vs a linear model on raw pixels
probe_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True)

def accuracy(net):
    net.eval()
    with torch.no_grad():
        return (net(X).argmax(1).cpu() == Y).float().mean().item()

def train_head(net, epochs=5):
    net.to(device).train()
    opt = torch.optim.Adam((p for p in net.parameters() if p.requires_grad), lr=1e-3)
    ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for xb, yb in probe_loader:
            xb = xb.view(xb.size(0), -1).to(device); yb = yb.to(device)
            loss = ce(net(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    return net

frozen_enc = copy.deepcopy(model.encoder)          # copy so the live model stays trainable
for p in frozen_enc.parameters():
    p.requires_grad = False                        # freeze: only the head learns
probe = nn.Sequential(frozen_enc, nn.Linear(32, 10))
raw   = nn.Linear(784, 10)                         # baseline: linear model straight on pixels

train_head(probe); train_head(raw)
p_acc, r_acc = accuracy(probe), accuracy(raw)
print(f"linear probe on frozen 32-d latent : {p_acc*100:.1f}% test acc")
print(f"linear model on raw 784 pixels     : {r_acc*100:.1f}% test acc")
print(f"-> the frozen-latent probe {'beats' if p_acc > r_acc else 'trails'} raw pixels here "
      f"(both on the same labels).")
print("The real payoff: the encoder was trained WITHOUT labels, so a probe needs only a few labels.")

## 8. From autoencoder to VAE (a first generative model)

A plain autoencoder **reconstructs** but can't **generate**: each input lands at one point `z`, so the latent is full of **holes** — a random `z` decodes to garbage. A **Variational Autoencoder (VAE)** fixes this: the encoder emits a **distribution** (a mean μ and variance σ²) per latent dim, you **sample** `z` before decoding, and a gentle pressure keeps every input's distribution **packed around the origin**. The holes close up, and now you can **sample a fresh `z` and decode brand-new data**. Same encoder→latent→decoder shape, with sampling at the bottleneck. The trade: slightly *blurrier* samples for a latent you can generate from and interpolate through. Lecture 14 places the VAE alongside GANs, diffusion, and autoregressive LLMs.

## 9. Pitfalls & misconceptions

- **Wrong loss for the data.** MSE for real-valued inputs, **BCE for 0/1** (pixel) data — we used `BCEWithLogitsLoss` above for exactly this reason.
- **"MSE = 0 means a great model."** No — an autoencoder hits 0 by learning the **identity** when dim(z)≥dim(x). Zero error is a **red flag**, not a trophy (unless the data truly lives on a lower-dimensional manifold, like the (a,2a) pairs).
- **"The bottleneck must be tiny."** Width is a **hyperparameter to tune** — wide enough to keep the signal, narrow enough to forbid copying.
- **Trusting the loss instead of probing.** Sharp reconstruction ≠ useful features. Always probe (cluster / linear-probe / downstream accuracy).
- **Forgetting to normalise inputs.** If features are on very different scales, MSE chases the big ones and the latent encodes *scale*, not structure — standardise first. (MNIST is already in [0,1], so it's safe here.)
- **Leaving the decoder in.** For a downstream task the prize is the **encoder** — freeze it (§7), throw the decoder away.
- **Cosine ignores magnitude.** [1,1] and [100,100] score 1.0. If scale carries meaning, use Euclidean distance.

## Practice

**P1 — reconstruction by hand.** Change `x` below so the two pairs are unequal in different ways and watch the width-2 MSE rise — the bottleneck can only keep each pair's **average**.

In [ ]:
x = torch.tensor([2., 4., 6., 10.])            # TODO: try e.g. [1., 9., 6., 6.] or [0., 10., 3., 3.]
pair_means = torch.tensor([x[:2].mean(), x[:2].mean(), x[2:].mean(), x[2:].mean()])
overall    = torch.full((4,), x.mean().item())
print(f"x              = {x.tolist()}")
print(f"width-2 x̂ (pair means)  = {pair_means.tolist()}   MSE = {((x-pair_means)**2).mean():.2f}")
print(f"width-1 x̂ (overall mean)= {overall.tolist()}   MSE = {((x-overall)**2).mean():.2f}")

**P2 — cosine on your own vectors.** Here are hand-made 3-feature "animal" embeddings. Add your own animal and check its nearest neighbour matches your intuition; try one that sits *between* two groups.

In [ ]:
animals = {
    "cat":    torch.tensor([0.9, 0.8, 0.1]),   # [furry, four-legged, aquatic]
    "dog":    torch.tensor([0.9, 0.9, 0.1]),
    "dolphin":torch.tensor([0.2, 0.0, 1.0]),
    "shark":  torch.tensor([0.0, 0.0, 1.0]),
    # TODO: add your own, e.g. "otter": torch.tensor([0.7, 0.4, 0.8]),
}
names = list(animals); V = torch.stack([animals[n] for n in names])
query = "cat"                                   # TODO: change the query
qv = animals[query]
sims = F.cosine_similarity(qv.unsqueeze(0), V, dim=1)
order = sims.argsort(descending=True)
print(f"nearest to '{query}' by cosine similarity:")
for i in order:
    if names[i] != query:
        print(f"  {names[i]:8} {sims[i]:.3f}")

**P3 (paper).** Sketch an autoencoder for 28×28 images: what are `dim(x)` and a sensible `dim(z)`? What loss, and why no labels? *(dim(x)=784; dim(z) maybe 16–64 — below 784 to force compression; BCE for [0,1] pixels; no labels because the target is the input itself — self-supervised. This is the embedding side HW6 grades.)*

**P4 (optional).** You already built the frozen-encoder **linear probe** in §7 — try unfreezing the encoder with a **10× smaller** learning rate than the head and see whether accuracy improves (the transfer-learning trade-off from Lecture 8).

## Key terms
- **Encoder / Decoder** — map `x`→`z` (compress) and `z`→`x̂` (reconstruct).
- **Latent / representation / code** — the compressed vector `z` at the bottleneck.
- **Bottleneck** — the narrow layer (dim(z)<dim(x)) that forbids copying and forces compression.
- **Autoencoder** — encoder-decoder trained to reconstruct its own input (**self-supervised**).
- **VAE** — encoder emits a *distribution* over `z`, kept packed around the origin, so you can **sample and generate**.
- **Embedding** — a latent where similar inputs are geometrically close.
- **Cosine similarity** — cosine of the angle between vectors; closeness that ignores magnitude.
- **Linear probe** — a single linear layer on a *frozen* representation, to test its usefulness.